In [1]:
#!/usr/bin/env python3
"""
=============================================================================
SCRIPT 3: IT→IA TRANSITION + IA vs AR — ALL LINEAGES, DONOR-LEVEL
=============================================================================
Key questions:
  1. IT→IA: Does "brake release" actually happen? (donor-level)
  2. IT→IA: What changes in each lineage?
  3. IA vs AR: What distinguishes failed clearance from success?
  4. Cross-lineage: Which changes are lineage-specific vs pan-immune?

Also: AIM2/NLRC4/MEFV cross-lineage verification (from myeloid findings)
=============================================================================
"""
!pip install -q scanpy leidenalg pyscenic gseapy openpyxl

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import scipy.sparse as sp
import os
import warnings
warnings.filterwarnings('ignore')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 605.3/605.3 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 133.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 125.4 MB/s eta 0:00:00
   ━━━━━━

In [3]:
# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ============================================================
# SETUP
# ============================================================
DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
SAVE_DIR = '/content/drive/MyDrive/ITLAS/results/task3_donor_validation/figures'
os.makedirs(SAVE_DIR, exist_ok=True)

print("Loading dataset...")
adata = sc.read_h5ad(DATA_PATH)
print(f"Dataset: {adata.shape[0]} cells, {adata.shape[1]} genes")


Loading dataset...
Dataset: 243000 cells, 24452 genes


In [5]:
STAGE_ORDER = ['NL', 'IT', 'IA', 'AR', 'CR']
STAGE_COLORS = {'NL': 'forestgreen', 'IT': 'crimson', 'IA': 'darkorange',
                'AR': 'royalblue', 'CR': 'purple'}
LINEAGES = ['NK', 'CD8_T', 'CD4_T', 'Myeloid', 'B', 'PlasmaB']
np.random.seed(42)

In [6]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================
def get_donor_means(adata_sub, gene_name):
    if gene_name not in adata_sub.var_names:
        return None
    if sp.issparse(adata_sub.X):
        expr = adata_sub[:, gene_name].X.toarray().flatten()
    else:
        expr = adata_sub[:, gene_name].X.flatten()
    df = pd.DataFrame({
        'expression': expr,
        'Stage': adata_sub.obs['Stage'].astype(str).values,
        'donor_id': adata_sub.obs['donor_id'].astype(str).values
    })
    return df.groupby(['donor_id', 'Stage'])['expression'].mean().reset_index()

def donor_test(donor_df, stage_a, stage_b):
    va = donor_df[donor_df['Stage'] == stage_a]['expression'].values
    vb = donor_df[donor_df['Stage'] == stage_b]['expression'].values
    if len(va) < 2 or len(vb) < 2:
        return {'va': va, 'vb': vb, 'p': 1.0, 'direction': 'NS',
                'consist': 0, 'total': 0, 'change': 0}
    _, p_gt = mannwhitneyu(va, vb, alternative='greater')
    _, p_lt = mannwhitneyu(va, vb, alternative='less')
    if p_gt < p_lt:
        consist = sum(1 for a in va for b in vb if a > b)
        direction = f'{stage_a}>{stage_b}'
        p_use = p_gt
    else:
        consist = sum(1 for a in va for b in vb if a < b)
        direction = f'{stage_b}>{stage_a}'
        p_use = p_lt
    total = len(va) * len(vb)
    change = (vb.mean() - va.mean()) / va.mean() * 100 if va.mean() > 0 else float('inf')
    return {'va': va, 'vb': vb, 'p': p_use, 'direction': direction,
            'consist': consist, 'total': total, 'change': change}


In [7]:
# 🔴 CRITICAL: 'processed' subfolder 포함 확인!
PROJECT_ROOT = '/content/drive/MyDrive/ITLAS'
DATA_DIR     = os.path.join(PROJECT_ROOT, 'data', 'processed')   # ← processed 빠뜨리지 말 것
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
FIGURES_DIR  = os.path.join(RESULTS_DIR, 'figures')
TABLES_DIR   = os.path.join(RESULTS_DIR, 'tables')

# h5ad 파일 경로
H5AD_PATH = os.path.join(DATA_DIR, 'GSE182159_gut2021_annotated.h5ad')

# --- Step 4: Verify paths exist ---
print("=" * 60)
print("PATH VERIFICATION")
print("=" * 60)

paths_to_check = {
    'PROJECT_ROOT': PROJECT_ROOT,
    'DATA_DIR':     DATA_DIR,
    'RESULTS_DIR':  RESULTS_DIR,
    'H5AD_PATH':    H5AD_PATH,
}

all_ok = True
for name, path in paths_to_check.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌ NOT FOUND"
    print(f"  {status}  {name:15s} = {path}")
    if not exists:
        all_ok = False

# --- Step 5: If h5ad not found, search for it ---
if not os.path.exists(H5AD_PATH):
    print("\n⚠️  H5AD file not found at expected path. Searching...")
    import glob
    candidates = glob.glob(os.path.join(PROJECT_ROOT, '**', '*.h5ad'), recursive=True)
    if candidates:
        for c in candidates:
            size_gb = os.path.getsize(c) / (1024**3)
            print(f"  Found: {c} ({size_gb:.2f} GB)")
        print("\n  → H5AD_PATH를 위 경로 중 하나로 수정하세요.")
    else:
        print("  ❌ No .h5ad files found in ITLAS folder!")

# --- Step 6: Create output directories ---
for d in [RESULTS_DIR, FIGURES_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

if all_ok:
    print("\n✅ 모든 경로 확인 완료. 다음 셀로 진행 가능.")
else:
    print("\n🔴 경로 문제 해결 후 다음 셀로 진행하세요.")

PATH VERIFICATION
  ✅  PROJECT_ROOT    = /content/drive/MyDrive/ITLAS
  ✅  DATA_DIR        = /content/drive/MyDrive/ITLAS/data/processed
  ✅  RESULTS_DIR     = /content/drive/MyDrive/ITLAS/results
  ✅  H5AD_PATH       = /content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad

✅ 모든 경로 확인 완료. 다음 셀로 진행 가능.


In [8]:
# ============================================================
# 00_ITLAS_Setup.ipynb — Cell 3
# Donor ID Extraction + AUCell Score File Search
# ============================================================

# ═══════════════════════════════════════════════════════════
# PART A: Donor ID 추출
# ═══════════════════════════════════════════════════════════
# sample 값 예: 'GSM5519467_P190604_Blood_1'
# → Patient ID = 'P190604' (두 번째 필드)

# 먼저 sample 값 구조 확인
print("=" * 60)
print("PART A: DONOR ID EXTRACTION")
print("=" * 60)
print("\nSample values (처음 10개):")
for val in adata.obs['sample'].unique()[:10]:
    parts = str(val).split('_')
    print(f"  {val}")
    print(f"    → parts: {parts}")

# Patient ID 추출 시도
# 패턴: GSM5519467_P190604_Blood_1 → P190604
adata.obs['donor_id'] = adata.obs['sample'].astype(str).apply(
    lambda x: x.split('_')[1] if len(x.split('_')) >= 2 else x
)

n_donors = adata.obs['donor_id'].nunique()
print(f"\n추출된 donor_id unique 수: {n_donors}")
print(f"기대값: 23")

if n_donors == 23:
    print("✅ 23 donors 정확히 추출됨!")
else:
    print(f"⚠️  {n_donors}명 → 수동 확인 필요")

# Donor × Stage 교차표
print("\n📊 Donor × Stage 교차표 (cell counts):")
ct = pd.crosstab(adata.obs['donor_id'], adata.obs['Stage'])
# Stage 순서 정렬
stage_order = ['NL', 'IT', 'IA', 'AR', 'CR']
ct = ct.reindex(columns=[s for s in stage_order if s in ct.columns])
print(ct.to_string())

# 각 Stage별 donor 수
print("\n📊 Donors per Stage:")
for stage in stage_order:
    donors_in_stage = adata.obs[adata.obs['Stage'] == stage]['donor_id'].nunique()
    print(f"   {stage}: n = {donors_in_stage}")

# ═══════════════════════════════════════════════════════════
# PART B: AUCell 26 Pathway 점수 파일 탐색
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PART B: AUCell SCORE FILE SEARCH")
print("=" * 60)

import glob

# 검색 패턴: AUCell 관련 파일
search_patterns = [
    os.path.join(PROJECT_ROOT, '**', '*aucell*'),
    os.path.join(PROJECT_ROOT, '**', '*AUCell*'),
    os.path.join(PROJECT_ROOT, '**', '*pathway*score*'),
    os.path.join(PROJECT_ROOT, '**', '*26_pathway*'),
    os.path.join(PROJECT_ROOT, '**', '*inflammasome*'),
    os.path.join(PROJECT_ROOT, 'results', '**', '*.csv'),
    os.path.join(PROJECT_ROOT, 'results', '**', '*.pkl'),
    os.path.join(PROJECT_ROOT, 'results', '**', '*.parquet'),
    os.path.join(PROJECT_ROOT, 'data', '**', '*.csv'),
]

found_files = set()
for pattern in search_patterns:
    for f in glob.glob(pattern, recursive=True):
        found_files.add(f)

if found_files:
    print(f"\n발견된 관련 파일 ({len(found_files)}개):")
    for f in sorted(found_files):
        size_mb = os.path.getsize(f) / (1024**2)
        print(f"  {f} ({size_mb:.1f} MB)")
else:
    print("\n❌ AUCell 관련 파일을 찾지 못했습니다.")

# 혹시 다른 h5ad 파일에 AUCell이 포함되어 있을 수 있음
print("\n📋 다른 h5ad 파일 탐색:")
h5ad_files = glob.glob(os.path.join(PROJECT_ROOT, '**', '*.h5ad'), recursive=True)
for f in h5ad_files:
    size_gb = os.path.getsize(f) / (1024**3)
    print(f"  {f} ({size_gb:.2f} GB)")

# adata.obsm 확인 (AUCell이 여기 저장되었을 수 있음)
print(f"\n📋 adata.obsm keys: {list(adata.obsm.keys())}")

# adata.uns 확인
print(f"📋 adata.uns keys (상위): {list(adata.uns.keys())[:20]}")

# ═══════════════════════════════════════════════════════════
# PART C: 현재 상태 요약
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("SUMMARY: 다음 단계 결정을 위한 현재 상태")
print("=" * 60)

available_pw = [c for c in adata.obs.columns if c.startswith('PW_')]
print(f"\n현재 obs에 있는 PW 컬럼 ({len(available_pw)}개): {available_pw}")

missing_pw = [
    'Inflammasome', 'Immune_Evasion', 'Checkpoint', 'Exhaustion',
    'NK_Function', 'Senescence', 'Cancer_Associated', 'Cytotoxicity',
    'Treg', 'Naive_T', 'Memory_T', 'TF_Programs', 'Tissue_Resident',
    'Stemness', 'Apoptosis', 'Fibrosis', 'Epigenetics', 'Angiogenesis',
    'Cell_cycle_Proliferation', 'MITO_DYSFUNCTION', 'Metabolic_Recovery'
]
print(f"누락된 핵심 pathway ({len(missing_pw)}개): {missing_pw[:8]}...")
print(f"\n⚠️  결과를 확인 후 알려주세요:")
print(f"   1) Donor ID가 23명 맞는지")
print(f"   2) AUCell 파일이 어디에 있는지 (또는 재계산 필요한지)")

PART A: DONOR ID EXTRACTION

Sample values (처음 10개):
  GSM5519467_P190604_Blood_1
    → parts: ['GSM5519467', 'P190604', 'Blood', '1']
  GSM5519468_P190604_Blood_2
    → parts: ['GSM5519468', 'P190604', 'Blood', '2']
  GSM5519469_P190604_Liver_1
    → parts: ['GSM5519469', 'P190604', 'Liver', '1']
  GSM5519470_P190326_Blood_1
    → parts: ['GSM5519470', 'P190326', 'Blood', '1']
  GSM5519471_P190326_Liver_1
    → parts: ['GSM5519471', 'P190326', 'Liver', '1']
  GSM5519472_P190402_Liver_1
    → parts: ['GSM5519472', 'P190402', 'Liver', '1']
  GSM5519473_P190716_Blood_1
    → parts: ['GSM5519473', 'P190716', 'Blood', '1']
  GSM5519474_P190716_Blood_2
    → parts: ['GSM5519474', 'P190716', 'Blood', '2']
  GSM5519475_P190716_Liver_1
    → parts: ['GSM5519475', 'P190716', 'Liver', '1']
  GSM5519476_P190719_Blood_1
    → parts: ['GSM5519476', 'P190719', 'Blood', '1']

추출된 donor_id unique 수: 23
기대값: 23
✅ 23 donors 정확히 추출됨!

📊 Donor × Stage 교차표 (cell counts):
Stage        NL     IT     IA     A

In [9]:
# ============================================================
# 00_ITLAS_Setup.ipynb — Cell 2
# Load h5ad & Explore obs columns
# ============================================================
# 목적: DONOR_COL, STAGE_COL, LINEAGE_COL, SUBCLUSTER_COL 정확한 이름 확인
# ⚠️ KeyError: None 방지 — 이 셀에서 확인 후 Cell 3에서 확정

# --- Step 1: Load data ---
print("Loading h5ad... (1-2분 소요)")
adata = sc.read_h5ad(H5AD_PATH)
print(f"✅ Loaded: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")

# --- Step 2: ALL obs columns 전체 출력 ---
print("\n" + "=" * 70)
print("ALL COLUMNS IN adata.obs")
print("=" * 70)
for i, col in enumerate(adata.obs.columns.tolist()):
    nuniq = adata.obs[col].nunique()
    dtype = str(adata.obs[col].dtype)
    # 샘플값 (문자열로 변환하여 안전하게)
    samples = [str(x) for x in adata.obs[col].unique()[:5]]
    print(f"  {i+1:3d}. {col:35s} | {dtype:12s} | unique={nuniq:5d} | {samples}")

# --- Step 3: Stage/Group 컬럼 자동 탐색 ---
print("\n" + "=" * 70)
print("AUTO-DETECT: STAGE / GROUP COLUMN")
print("=" * 70)
for col in adata.obs.columns:
    vals = set(adata.obs[col].astype(str).unique())
    # NL, IT, IA, AR 중 3개 이상 포함하면 stage 컬럼일 가능성 높음
    stage_markers = {'NL', 'IT', 'IA', 'AR'}
    overlap = vals & stage_markers
    if len(overlap) >= 3:
        print(f"  🎯 '{col}' → contains {overlap}")
        print(f"     All values: {sorted(vals)}")
        # 🔴 CR vs AC 확인!
        if 'AC' in vals:
            print(f"     ⚠️  'AC' detected! (원본 데이터 = AC, 우리 논문 = CR)")
        if 'CR' in vals:
            print(f"     ✅ 'CR' confirmed (이미 변환됨)")

# --- Step 4: Donor/Sample 컬럼 자동 탐색 ---
print("\n" + "=" * 70)
print("AUTO-DETECT: DONOR / SAMPLE COLUMN")
print("=" * 70)
for col in adata.obs.columns:
    nuniq = adata.obs[col].nunique()
    # 23 donors 또는 그 근처
    if 15 <= nuniq <= 30:
        print(f"  🔍 '{col}' → {nuniq} unique values (23 donors expected)")
        print(f"     Sample: {adata.obs[col].unique()[:8].tolist()}")

# --- Step 5: Lineage / Celltype 컬럼 자동 탐색 ---
print("\n" + "=" * 70)
print("AUTO-DETECT: LINEAGE / CELLTYPE COLUMN")
print("=" * 70)
for col in adata.obs.columns:
    nuniq = adata.obs[col].nunique()
    vals_str = [str(x).lower() for x in adata.obs[col].unique()[:20]]
    vals_joined = ' '.join(vals_str)
    # 7-8 lineages 또는 59-60 subclusters
    if 5 <= nuniq <= 12 and any(k in vals_joined for k in ['nk', 'myeloid', 'cd4', 'cd8', 'b ']):
        print(f"  🎯 LINEAGE: '{col}' → {nuniq} values")
        print(f"     Values: {sorted(adata.obs[col].unique().tolist())}")
    if 55 <= nuniq <= 65:
        print(f"  🎯 SUBCLUSTER: '{col}' → {nuniq} values")
        print(f"     Sample: {adata.obs[col].unique()[:10].tolist()}")

# --- Step 6: AUCell score 컬럼 탐색 ---
print("\n" + "=" * 70)
print("AUTO-DETECT: AUCell PATHWAY SCORE COLUMNS")
print("=" * 70)
aucell_cols = []
for col in adata.obs.columns:
    if adata.obs[col].dtype in ['float32', 'float64'] and adata.obs[col].nunique() > 100:
        aucell_cols.append(col)

if aucell_cols:
    print(f"  Found {len(aucell_cols)} numeric columns (potential AUCell scores):")
    for col in aucell_cols[:30]:
        mean_val = adata.obs[col].mean()
        print(f"    {col:40s} | mean={mean_val:.4f}")
else:
    print("  ❌ No obvious AUCell columns found in obs")
    print("  → adata.obsm keys:", list(adata.obsm.keys()) if adata.obsm else "empty")

print("\n" + "=" * 70)
print("⚠️  위 출력을 확인하고 Cell 3에서 정확한 컬럼명을 확정합니다.")
print("=" * 70)

Loading h5ad... (1-2분 소요)
✅ Loaded: 243,000 cells × 24,452 genes

ALL COLUMNS IN adata.obs
    1. sample                              | category     | unique=   46 | ['GSM5519467_P190604_Blood_1', 'GSM5519468_P190604_Blood_2', 'GSM5519469_P190604_Liver_1', 'GSM5519470_P190326_Blood_1', 'GSM5519471_P190326_Liver_1']
    2. tissue                              | category     | unique=    2 | ['Blood', 'Liver']
    3. Stage                               | category     | unique=    5 | ['IT', 'AR', 'IA', 'NL', 'CR']
    4. IT_cluster_21                       | float64      | unique=242960 | ['3.7056410722019946', '4.034355906879201', '3.1085649192427978', '3.244250415808067', '4.0003165929609334']
    5. IT_cluster_23                       | float64      | unique=240984 | ['3.262586739225295', '3.6219994132958573', '2.7674797842803516', '2.320434523091733', '3.5876710363962117']
    6. IT_cluster_25                       | float64      | unique=114299 | ['4.199094219154806', '0.0', '1.71414

In [10]:
# ============================================================
# 00_ITLAS_Setup.ipynb — Cell 3
# Donor ID Extraction + AUCell Score File Search
# ============================================================

# ═══════════════════════════════════════════════════════════
# PART A: Donor ID 추출
# ═══════════════════════════════════════════════════════════
# sample 값 예: 'GSM5519467_P190604_Blood_1'
# → Patient ID = 'P190604' (두 번째 필드)

# 먼저 sample 값 구조 확인
print("=" * 60)
print("PART A: DONOR ID EXTRACTION")
print("=" * 60)
print("\nSample values (처음 10개):")
for val in adata.obs['sample'].unique()[:10]:
    parts = str(val).split('_')
    print(f"  {val}")
    print(f"    → parts: {parts}")

# Patient ID 추출 시도
# 패턴: GSM5519467_P190604_Blood_1 → P190604
adata.obs['donor_id'] = adata.obs['sample'].astype(str).apply(
    lambda x: x.split('_')[1] if len(x.split('_')) >= 2 else x
)

n_donors = adata.obs['donor_id'].nunique()
print(f"\n추출된 donor_id unique 수: {n_donors}")
print(f"기대값: 23")

if n_donors == 23:
    print("✅ 23 donors 정확히 추출됨!")
else:
    print(f"⚠️  {n_donors}명 → 수동 확인 필요")

# Donor × Stage 교차표
print("\n📊 Donor × Stage 교차표 (cell counts):")
ct = pd.crosstab(adata.obs['donor_id'], adata.obs['Stage'])
# Stage 순서 정렬
stage_order = ['NL', 'IT', 'IA', 'AR', 'CR']
ct = ct.reindex(columns=[s for s in stage_order if s in ct.columns])
print(ct.to_string())

# 각 Stage별 donor 수
print("\n📊 Donors per Stage:")
for stage in stage_order:
    donors_in_stage = adata.obs[adata.obs['Stage'] == stage]['donor_id'].nunique()
    print(f"   {stage}: n = {donors_in_stage}")

# ═══════════════════════════════════════════════════════════
# PART B: AUCell 26 Pathway 점수 파일 탐색
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PART B: AUCell SCORE FILE SEARCH")
print("=" * 60)

import glob

# 검색 패턴: AUCell 관련 파일
search_patterns = [
    os.path.join(PROJECT_ROOT, '**', '*aucell*'),
    os.path.join(PROJECT_ROOT, '**', '*AUCell*'),
    os.path.join(PROJECT_ROOT, '**', '*pathway*score*'),
    os.path.join(PROJECT_ROOT, '**', '*26_pathway*'),
    os.path.join(PROJECT_ROOT, '**', '*inflammasome*'),
    os.path.join(PROJECT_ROOT, 'results', '**', '*.csv'),
    os.path.join(PROJECT_ROOT, 'results', '**', '*.pkl'),
    os.path.join(PROJECT_ROOT, 'results', '**', '*.parquet'),
    os.path.join(PROJECT_ROOT, 'data', '**', '*.csv'),
]

found_files = set()
for pattern in search_patterns:
    for f in glob.glob(pattern, recursive=True):
        found_files.add(f)

if found_files:
    print(f"\n발견된 관련 파일 ({len(found_files)}개):")
    for f in sorted(found_files):
        size_mb = os.path.getsize(f) / (1024**2)
        print(f"  {f} ({size_mb:.1f} MB)")
else:
    print("\n❌ AUCell 관련 파일을 찾지 못했습니다.")

# 혹시 다른 h5ad 파일에 AUCell이 포함되어 있을 수 있음
print("\n📋 다른 h5ad 파일 탐색:")
h5ad_files = glob.glob(os.path.join(PROJECT_ROOT, '**', '*.h5ad'), recursive=True)
for f in h5ad_files:
    size_gb = os.path.getsize(f) / (1024**3)
    print(f"  {f} ({size_gb:.2f} GB)")

# adata.obsm 확인 (AUCell이 여기 저장되었을 수 있음)
print(f"\n📋 adata.obsm keys: {list(adata.obsm.keys())}")

# adata.uns 확인
print(f"📋 adata.uns keys (상위): {list(adata.uns.keys())[:20]}")

# ═══════════════════════════════════════════════════════════
# PART C: 현재 상태 요약
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("SUMMARY: 다음 단계 결정을 위한 현재 상태")
print("=" * 60)

available_pw = [c for c in adata.obs.columns if c.startswith('PW_')]
print(f"\n현재 obs에 있는 PW 컬럼 ({len(available_pw)}개): {available_pw}")

missing_pw = [
    'Inflammasome', 'Immune_Evasion', 'Checkpoint', 'Exhaustion',
    'NK_Function', 'Senescence', 'Cancer_Associated', 'Cytotoxicity',
    'Treg', 'Naive_T', 'Memory_T', 'TF_Programs', 'Tissue_Resident',
    'Stemness', 'Apoptosis', 'Fibrosis', 'Epigenetics', 'Angiogenesis',
    'Cell_cycle_Proliferation', 'MITO_DYSFUNCTION', 'Metabolic_Recovery'
]
print(f"누락된 핵심 pathway ({len(missing_pw)}개): {missing_pw[:8]}...")
print(f"\n⚠️  결과를 확인 후 알려주세요:")
print(f"   1) Donor ID가 23명 맞는지")
print(f"   2) AUCell 파일이 어디에 있는지 (또는 재계산 필요한지)")

PART A: DONOR ID EXTRACTION

Sample values (처음 10개):
  GSM5519467_P190604_Blood_1
    → parts: ['GSM5519467', 'P190604', 'Blood', '1']
  GSM5519468_P190604_Blood_2
    → parts: ['GSM5519468', 'P190604', 'Blood', '2']
  GSM5519469_P190604_Liver_1
    → parts: ['GSM5519469', 'P190604', 'Liver', '1']
  GSM5519470_P190326_Blood_1
    → parts: ['GSM5519470', 'P190326', 'Blood', '1']
  GSM5519471_P190326_Liver_1
    → parts: ['GSM5519471', 'P190326', 'Liver', '1']
  GSM5519472_P190402_Liver_1
    → parts: ['GSM5519472', 'P190402', 'Liver', '1']
  GSM5519473_P190716_Blood_1
    → parts: ['GSM5519473', 'P190716', 'Blood', '1']
  GSM5519474_P190716_Blood_2
    → parts: ['GSM5519474', 'P190716', 'Blood', '2']
  GSM5519475_P190716_Liver_1
    → parts: ['GSM5519475', 'P190716', 'Liver', '1']
  GSM5519476_P190719_Blood_1
    → parts: ['GSM5519476', 'P190719', 'Blood', '1']

추출된 donor_id unique 수: 23
기대값: 23
✅ 23 donors 정확히 추출됨!

📊 Donor × Stage 교차표 (cell counts):
Stage        NL     IT     IA     A

In [11]:
# ============================================================
# PART 1: CROSS-LINEAGE GENE MATRIX — IT vs IA
# ============================================================
print("\n" + "="*80)
print("PART 1: CROSS-LINEAGE GENE MATRIX — IT vs IA")
print("       Does 'brake release' happen at IT→IA transition?")
print("="*80)

# All genes that matter for IT→IA narrative
IT_IA_GENES = [
    # Fire genes (confirmed in myeloid)
    'CASP1', 'PYCARD', 'AIM2', 'NLRC4', 'MEFV',
    # Fire genes (NS in myeloid but need cross-lineage check)
    'NLRP3', 'IL1B', 'GSDMD', 'IL18',
    # Brake genes
    'LGALS9', 'TGFB1', 'TNFAIP3', 'IDO1', 'IL1RN',
    # Checkpoint/exhaustion
    'PDCD1', 'LAG3', 'HAVCR2', 'TIGIT', 'TOX',
    # Effector
    'GZMB', 'PRF1', 'IFNG',
]

for gene in IT_IA_GENES:
    if gene not in adata.var_names:
        print(f"  {gene}: NOT IN DATASET — skipping")
        continue

    print(f"\n--- {gene}: IT vs IA across lineages ---")
    print(f"  {'Lineage':<10s} {'IT mean':>10s} {'IA mean':>10s} {'Change':>10s} "
          f"{'Consist':>10s} {'p-value':>8s}")
    print("  " + "-"*60)

    for lineage in LINEAGES:
        lin_mask = adata.obs['major_lineage'] == lineage
        adata_lin = adata[lin_mask]
        donor_df = get_donor_means(adata_lin, gene)
        if donor_df is None:
            continue
        r = donor_test(donor_df, 'IT', 'IA')
        sig = " **" if r['p'] < 0.05 else " *" if r['p'] < 0.1 else ""
        ch = f"{r['change']:+.1f}%" if abs(r['change']) < 9999 else "INF"
        print(f"  {lineage:<10s} {r['va'].mean():>10.4f} {r['vb'].mean():>10.4f} "
              f"{ch:>10s} {r['consist']}/{r['total']:>7} {r['p']:>8.3f}{sig}")



PART 1: CROSS-LINEAGE GENE MATRIX — IT vs IA
       Does 'brake release' happen at IT→IA transition?

--- CASP1: IT vs IA across lineages ---
  Lineage       IT mean    IA mean     Change    Consist  p-value
  ------------------------------------------------------------
  NK             0.4219     0.3758     -10.9% 20/     30    0.214
  CD8_T          0.3482     0.3447      -1.0% 17/     30    0.396
  CD4_T          0.3561     0.3433      -3.6% 17/     30    0.396
  Myeloid        1.1295     1.1687      +3.5% 17/     30    0.396
  B              0.2377     0.2049     -13.8% 20/     30    0.214
  PlasmaB        0.0759     0.0679     -10.5% 17/     30    0.396

--- PYCARD: IT vs IA across lineages ---
  Lineage       IT mean    IA mean     Change    Consist  p-value
  ------------------------------------------------------------
  NK             0.4225     0.4420      +4.6% 16/     30    0.465
  CD8_T          0.2921     0.3268     +11.9% 20/     30    0.214
  CD4_T          0.2947     0

In [12]:
# ============================================================
# PART 2: CROSS-LINEAGE GENE MATRIX — IA vs AR
# ============================================================
print("\n" + "="*80)
print("PART 2: CROSS-LINEAGE GENE MATRIX — IA vs AR")
print("       What distinguishes failed vs successful clearance?")
print("="*80)

IA_AR_GENES = [
    # Inflammasome
    'CASP1', 'PYCARD', 'AIM2', 'NLRC4', 'MEFV', 'NLRP3', 'IL1B', 'GSDMD',
    # Brakes
    'LGALS9', 'TGFB1', 'TNFAIP3', 'IDO1', 'IL1RN',
    # Checkpoint
    'PDCD1', 'LAG3', 'HAVCR2', 'TIGIT', 'TOX',
    # Effector
    'GZMB', 'PRF1', 'IFNG', 'GNLY',
    # Exhaustion
    'LAYN', 'ENTPD1', 'BATF',
]

for gene in IA_AR_GENES:
    if gene not in adata.var_names:
        continue

    print(f"\n--- {gene}: IA vs AR across lineages ---")
    print(f"  {'Lineage':<10s} {'IA mean':>10s} {'AR mean':>10s} {'Change':>10s} "
          f"{'Consist':>10s} {'p-value':>8s}")
    print("  " + "-"*60)

    for lineage in LINEAGES:
        lin_mask = adata.obs['major_lineage'] == lineage
        adata_lin = adata[lin_mask]
        donor_df = get_donor_means(adata_lin, gene)
        if donor_df is None:
            continue
        r = donor_test(donor_df, 'IA', 'AR')
        sig = " **" if r['p'] < 0.05 else " *" if r['p'] < 0.1 else ""
        ch = f"{r['change']:+.1f}%" if abs(r['change']) < 9999 else "INF"
        print(f"  {lineage:<10s} {r['va'].mean():>10.4f} {r['vb'].mean():>10.4f} "
              f"{ch:>10s} {r['consist']}/{r['total']:>7} {r['p']:>8.3f}{sig}")



PART 2: CROSS-LINEAGE GENE MATRIX — IA vs AR
       What distinguishes failed vs successful clearance?

--- CASP1: IA vs AR across lineages ---
  Lineage       IA mean    AR mean     Change    Consist  p-value
  ------------------------------------------------------------
  NK             0.3758     0.4907     +30.6% 15/     15    0.018 **
  CD8_T          0.3447     0.3907     +13.3% 10/     15    0.286
  CD4_T          0.3433     0.3961     +15.4% 10/     15    0.286
  Myeloid        1.1687     1.1099      -5.0% 8/     15    0.500
  B              0.2049     0.2576     +25.7% 14/     15    0.036 **
  PlasmaB        0.0679     0.1190     +75.3% 12/     15    0.125

--- PYCARD: IA vs AR across lineages ---
  Lineage       IA mean    AR mean     Change    Consist  p-value
  ------------------------------------------------------------
  NK             0.4420     0.5966     +35.0% 13/     15    0.071 *
  CD8_T          0.3268     0.5057     +54.7% 11/     15    0.196
  CD4_T          0.2

In [13]:
# ============================================================
# PART 3: AIM2/NLRC4/MEFV CROSS-LINEAGE — Myeloid-specific?
# ============================================================
print("\n" + "="*80)
print("PART 3: AIM2/NLRC4/MEFV — CROSS-LINEAGE NL vs IT")
print("       Are these Myeloid-specific activations?")
print("="*80)

for gene in ['AIM2', 'NLRC4', 'MEFV']:
    if gene not in adata.var_names:
        print(f"  {gene}: NOT FOUND")
        continue

    print(f"\n--- {gene}: NL vs IT across lineages ---")
    print(f"  {'Lineage':<10s} {'NL mean':>10s} {'IT mean':>10s} {'Change':>10s} "
          f"{'Consist':>10s} {'p-value':>8s}")
    print("  " + "-"*60)

    for lineage in LINEAGES:
        lin_mask = adata.obs['major_lineage'] == lineage
        adata_lin = adata[lin_mask]
        donor_df = get_donor_means(adata_lin, gene)
        if donor_df is None:
            continue
        r = donor_test(donor_df, 'NL', 'IT')
        sig = " **" if r['p'] < 0.05 else " *" if r['p'] < 0.1 else ""
        ch = f"{r['change']:+.1f}%" if abs(r['change']) < 9999 else "INF"
        print(f"  {lineage:<10s} {r['va'].mean():>10.4f} {r['vb'].mean():>10.4f} "
              f"{ch:>10s} {r['consist']}/{r['total']:>7} {r['p']:>8.3f}{sig}")


PART 3: AIM2/NLRC4/MEFV — CROSS-LINEAGE NL vs IT
       Are these Myeloid-specific activations?

--- AIM2: NL vs IT across lineages ---
  Lineage       NL mean    IT mean     Change    Consist  p-value
  ------------------------------------------------------------
  NK             0.0126     0.0132      +4.9% 19/     36    0.469
  CD8_T          0.0347     0.0341      -1.8% 18/     36    0.531
  CD4_T          0.0243     0.0289     +19.1% 21/     36    0.344
  Myeloid        0.0118     0.0625    +430.7% 33/     36    0.010 **
  B              0.1937     0.3970    +105.0% 32/     36    0.013 **
  PlasmaB        0.1162     0.1369     +17.8% 23/     36    0.242

--- NLRC4: NL vs IT across lineages ---
  Lineage       NL mean    IT mean     Change    Consist  p-value
  ------------------------------------------------------------
  NK             0.0000     0.0002        INF 6/     36    0.202
  CD8_T          0.0000     0.0002        INF 6/     36    0.202
  CD4_T          0.0031     0.00

In [14]:
# ============================================================
# PART 4: PYCARD/CASP1 CROSS-LINEAGE — Myeloid-specific?
# ============================================================
print("\n" + "="*80)
print("PART 4: PYCARD/CASP1 — CROSS-LINEAGE NL vs IT")
print("="*80)

for gene in ['PYCARD', 'CASP1']:
    if gene not in adata.var_names:
        continue

    print(f"\n--- {gene}: NL vs IT across lineages ---")
    print(f"  {'Lineage':<10s} {'NL mean':>10s} {'IT mean':>10s} {'Change':>10s} "
          f"{'Consist':>10s} {'p-value':>8s}")
    print("  " + "-"*60)

    for lineage in LINEAGES:
        lin_mask = adata.obs['major_lineage'] == lineage
        adata_lin = adata[lin_mask]
        donor_df = get_donor_means(adata_lin, gene)
        if donor_df is None:
            continue
        r = donor_test(donor_df, 'NL', 'IT')
        sig = " **" if r['p'] < 0.05 else " *" if r['p'] < 0.1 else ""
        ch = f"{r['change']:+.1f}%" if abs(r['change']) < 9999 else "INF"
        print(f"  {lineage:<10s} {r['va'].mean():>10.4f} {r['vb'].mean():>10.4f} "
              f"{ch:>10s} {r['consist']}/{r['total']:>7} {r['p']:>8.3f}{sig}")


PART 4: PYCARD/CASP1 — CROSS-LINEAGE NL vs IT

--- PYCARD: NL vs IT across lineages ---
  Lineage       NL mean    IT mean     Change    Consist  p-value
  ------------------------------------------------------------
  NK             0.2708     0.4225     +56.0% 29/     36    0.047 **
  CD8_T          0.2396     0.2921     +21.9% 26/     36    0.120
  CD4_T          0.2517     0.2947     +17.1% 22/     36    0.294
  Myeloid        1.4325     1.7280     +20.6% 30/     36    0.032 **
  B              0.2411     0.4503     +86.7% 32/     36    0.013 **
  PlasmaB        0.2875     0.8234    +186.3% 36/     36    0.001 **

--- CASP1: NL vs IT across lineages ---
  Lineage       NL mean    IT mean     Change    Consist  p-value
  ------------------------------------------------------------
  NK             0.3506     0.4219     +20.3% 23/     36    0.242
  CD8_T          0.3282     0.3482      +6.1% 18/     36    0.531
  CD4_T          0.3161     0.3561     +12.7% 25/     36    0.155
  Myel

In [15]:
# ============================================================
# PART 5: TGFB1/LGALS9 FULL TRAJECTORY — MYELOID (5-stage)
# ============================================================
print("\n" + "="*80)
print("PART 5: LGALS9 & TGFB1 — FULL PAIRWISE (MYELOID)")
print("       Complete trajectory: NL→IT→IA→AR→CR")
print("="*80)

myeloid_mask = adata.obs['major_lineage'] == 'Myeloid'
adata_myeloid = adata[myeloid_mask]

for gene in ['LGALS9', 'TGFB1']:
    donor_df = get_donor_means(adata_myeloid, gene)
    if donor_df is None:
        continue

    print(f"\n--- {gene} (Myeloid) — All pairwise ---")
    print(f"  Stage means:")
    for stage in STAGE_ORDER:
        sd = donor_df[donor_df['Stage'] == stage]
        if len(sd) > 0:
            print(f"    {stage}: {sd['expression'].mean():.4f} (n={len(sd)})")

    all_pairs = [('NL','IT'), ('IT','IA'), ('IA','AR'), ('IA','CR'),
                 ('NL','IA'), ('NL','CR'), ('AR','CR')]
    print(f"\n  {'Comparison':<12s} {'Change':>10s} {'Consist':>10s} {'p-value':>8s}")
    print("  " + "-"*45)
    for sa, sb in all_pairs:
        r = donor_test(donor_df, sa, sb)
        sig = " **" if r['p'] < 0.05 else " *" if r['p'] < 0.1 else ""
        ch = f"{r['change']:+.1f}%" if abs(r['change']) < 9999 else "INF"
        print(f"  {sa}→{sb:<10s} {ch:>10s} {r['consist']}/{r['total']:>7} {r['p']:>8.3f}{sig}")



PART 5: LGALS9 & TGFB1 — FULL PAIRWISE (MYELOID)
       Complete trajectory: NL→IT→IA→AR→CR

--- LGALS9 (Myeloid) — All pairwise ---
  Stage means:
    NL: 0.6694 (n=6)
    IT: 1.1703 (n=6)
    IA: 1.2478 (n=5)
    AR: 1.0463 (n=3)
    CR: 1.0612 (n=3)

  Comparison       Change    Consist  p-value
  ---------------------------------------------
  NL→IT             +74.8% 36/     36    0.001 **
  IT→IA              +6.6% 18/     30    0.331
  IA→AR             -16.2% 11/     15    0.196
  IA→CR             -15.0% 10/     15    0.286
  NL→IA             +86.4% 30/     30    0.002 **
  NL→CR             +58.5% 18/     18    0.012 **
  AR→CR              +1.4% 6/      9    0.350

--- TGFB1 (Myeloid) — All pairwise ---
  Stage means:
    NL: 0.8116 (n=6)
    IT: 1.2373 (n=6)
    IA: 1.4007 (n=5)
    AR: 1.2521 (n=3)
    CR: 1.1556 (n=3)

  Comparison       Change    Consist  p-value
  ---------------------------------------------
  NL→IT             +52.4% 33/     36    0.008 **
  IT→IA  

In [16]:
# ============================================================
# PART 6: COMPREHENSIVE SUMMARY — MASTER TABLE
# ============================================================
print("\n" + "="*80)
print("PART 6: MASTER VALIDATION SUMMARY")
print("       All confirmed findings (p<0.05) across all analyses")
print("="*80)

MASTER_GENES = [
    # Myeloid fire (confirmed)
    'CASP1', 'PYCARD', 'AIM2', 'NLRC4', 'MEFV',
    # Myeloid brake (confirmed)
    'LGALS9', 'TGFB1',
    # Myeloid fire (failed)
    'NLRP3', 'IL1B', 'GSDMD', 'IL18',
    # Myeloid brake (failed)
    'TNFAIP3', 'IDO1', 'IL1RN',
    # Cross-lineage
    'CASP8', 'FAS', 'IL2RG', 'MTOR', 'JAK1', 'TET2',
]

print(f"\n{'Gene':<10s} | ", end="")
for lin in LINEAGES:
    print(f"{lin:>10s}", end=" | ")
print()
print("-" * 85)

for gene in MASTER_GENES:
    if gene not in adata.var_names:
        continue
    print(f"  {gene:<10s}", end="")
    for lineage in LINEAGES:
        lin_mask = adata.obs['major_lineage'] == lineage
        adata_lin = adata[lin_mask]
        donor_df = get_donor_means(adata_lin, gene)
        if donor_df is None:
            print(f"{'N/A':>10s}", end=" | ")
            continue
        r = donor_test(donor_df, 'NL', 'IT')
        sig = "**" if r['p'] < 0.05 else "* " if r['p'] < 0.1 else "  "
        ch = f"{r['change']:+.0f}%" if abs(r['change']) < 9999 else "INF"
        print(f" {ch:>5s}{sig}", end="  | ")
    print()

print("\n" + "="*80)
print("ALL ANALYSES COMPLETE")
print("="*80)
print("\nTotal scripts run: 3/3")
print("Coverage: Myeloid + B + PlasmaB + NK + CD8_T + CD4_T")
print("Comparisons: NL vs IT, NL vs CR, IT vs IA, IA vs AR")
print("Method: Donor-level Mann-Whitney U with consistency counts")



PART 6: MASTER VALIDATION SUMMARY
       All confirmed findings (p<0.05) across all analyses

Gene       |         NK |      CD8_T |      CD4_T |    Myeloid |          B |    PlasmaB | 
-------------------------------------------------------------------------------------
  CASP1       +20%    |    +6%    |   +13%    |   +59%**  |   +33%    |   +50%    | 
  PYCARD      +56%**  |   +22%    |   +17%    |   +21%**  |   +87%**  |  +186%**  | 
  AIM2         +5%    |    -2%    |   +19%    |  +431%**  |  +105%**  |   +18%    | 
  NLRC4        INF    |    INF    |   -82%    |   +82%**  |  +103%*   |    INF    | 
  MEFV         INF*   |   -11%    |   -46%    |  +122%**  |    INF    |    INF    | 
  LGALS9      +43%**  |   +23%    |   +71%**  |   +75%**  |   +33%*   |  +525%**  | 
  TGFB1        -4%    |   -21%*   |   -23%*   |   +52%**  |   -10%    |   +65%**  | 
  NLRP3       -30%*   |   -47%    |   -42%    |    -4%    |    INF**  |    INF**  | 
  IL1B       +563%    |   +55%*   |  +294%    |